#### Backreferences [Tutorial]

A *capture group* of a regular expression is a subexpression enclosed in parentheses. Capture groups allow to apply operators over the entire group, as in `(a|b|c)?`.

Capture groups are also used for backreferences, which allow the content of a previously matched capture group to be reused for matching. This feature is supported in POSIX *basic regular expressions* (BRE), *Perl Compatible Regular Expression* (PCRE), and most programming language libraries; see below for exceptions.

For example, for matching words that repeat themselves, a *reduplication*, such as "bye-bye", is written as `([a-z]+)-\1`. Capture groups are numbered, so the first capture group`([a-z]+)` can be backreferenced using `\1`. Once `([a-z]+)` is matched, `-` is matched, and then the string matched by `([a-z]+)` is matched again with the backreference `\1`.

Here is the example in JavaScript. Regular expressions in JavaScript are written as `/.../`. The method `test` of a regular expression returns if the parameter matches the regular expression. In Jupyter notebooks, the method `element.append(s)` displays `s`.

In [ ]:
%%javascript
const re = /([a-z]+)-\1/
element.append(re.test('bye-bye'), ' ')
element.append(re.test('byebye'), ' ')
element.append(re.test('go-go'), ' ')
element.append(re.test('no-go'), ' ')

There are limitations on the number of capture groups. BRE allows 1 to 9 capture groups to be backreferenced, while PCRE supports up to 99 backreferences. In BRE, `\10` is interpreted as a reference to capture group 1 followed by a literal character 0. Regular expression libraries in programming languages (e.g. Python's `re` module) are often Perl-like and allow up to 99 capture groups.

##### Capture Group Numbering

Capture groups are numbered from where the *opening* parentheses occur in the regular expression. For example, the regular expression `(a(b(c)|(d)))` has four capture groups:

1. `a(b(c)|(d))`
2. `b(c)\|(d)`
3. `b(c)`
4. `(d)`

For the string match `abc`, capture group 1 matches `abc`, capture group 2 matches `bc` (nested within capture group 1), and capture group 3 matches *c* (nested within capture group 2).

|a(b(c)\|(d))|b(c)\|(d)|b(c)|(d)|
|--|--|--|--|
|abc|bc|c||

For the match `ad`, capture group 1 matches `ad`, capture group 2 matches `d`, and capture group 4 also matches `d`.

|a(b(c)\|(d))|b(c)\|(d)|b(c)|(d)|
|--|--|--|--|
|ad|d||d|

In JavaScript, regular expressions can also be matched by `string.match(regex)`, where the regular expressions can have a global (`g`) flag:
- With the `g` flag, all matches are returned.
- Without the `g` flag, only the first match and its capturing groups are returned.

A multi-line string is written by enclosing it in `` `...` ``:

In [ ]:
%%javascript
const string = `abc
ad`
const regex = /(a(b(c)|(d)))/        // first match
element.append(string.match(regex))

The result shows that `abc` is matched with capture groups `abc`, `bc`, and `c`.

In [ ]:
%%javascript
const string = `aacac
bbcbc`
const regex = /(a|b)(\1c)\2/g       // global match
element.append(string.match(regex))

In [ ]:
%%javascript
const string = `abc
ad`
const regex = /(a(b(c)|(d)))/g       // global match
element.append(string.match(regex))

The result shows that lines `abc` and `ad` are matched.

##### Nested Backreferences

Backreferences can be used within other capture groups. When a backreference is used within the capture group it references, it is called a *nested backreference.* For example, `(\1a\1|(bb))+` contains a nested backreference. It can be used to match strings like *bbbbabb* where *bb* is first matched and then made available for backreferencing, allowing *bbabb* to be matched by rule `\1a\1`.

In some regular expression libraries, the backreference option should be the first option among choices for it to be considered if the option starts with the backreference (e.g. `(\1a\1|(bb))+`). Otherwise, a greedy algorithm might overlook the backreference option. Nested backreferences can also be used in concatenations (e.g. `(a\1)+`).

In [ ]:
%%javascript
const re = /(\1a\1|bb)*/
element.append(re.test('bbbbabb'))

A nested backreference with concatenation is `(ab\1\1)\1c`: here `ab\1` matches `abab`, then `ab\1\1` matches `abababab`, making `abababab` the capture content.

In [ ]:
%%javascript
const re = /(ab\1\1)\1c/
element.append(re.test('ababababababababc'))

The regular expression `(ab)(\1c)\2` involves capturing content from another capture group for matching: 

In [ ]:
%%javascript
const re = /(ab)(\1c)\2/
element.append(re.test('ababcabc'), ' ')

The final example of a nested backreference is `(\1\1\1|ab)*`:

In [ ]:
%%javascript
const re = /(\1\1\1|ab)*/
element.append(re.test('abababab'), ' ')

##### Relative Backreferences

Most Perl-like regular expression engines provide syntactic sugar for *relative backreferences*. The syntax is `\g{-1}` in Perl and `k<-1>` in Ruby, where the `-1` refers to the first relative preceding capture group; we would use `-2` to refer to the second relative preceding capture group, and so on. As a nested backreference, `-1` refers to the current capturing group. For example, `([a-z]+)([a-z]+)\1` and `([a-z]+)([a-z]+)\g{-2}` are equivalent. Also, `(a+)\g{-1}(b+)\g{-1}` and `(a+)\1(b+)\2` are equivalent. The numbering is relative to the relative backreference.

##### Named Backreferences

Backreferences can be named by adding `?'name'` at the start of the capture group. The capture group is referenced using `\g{name}` in Perl. For example, `(?'word'[a-z]+)-\g{word}` is equivalent to `([a-z]+)-\1` for matching repeated words.

##### Regular Expression Standards

|   |Backreferences|Named capture groups|Named backreferences|Relative backreferences|
|:--|:--|:--|:--|:--|
|POSIX BRE| `\1` |-|-|-|
|POSIX ERE|-|-|-|-|
|PCRE| `\1`, `\g{1}` | `(?<name>...)`, `(?'name'...)`|`\g{name}`, `\k<name>` | `\g{-1}` |

Unix utilities like `sed` and `grep` are BRE-compliant by default. POSIX Extended regular expressions (ERE) can be used in `grep` with the `-e` option, but ERE does not support backreferences because [ERE maintains regularity](https://unix.stackexchange.com/questions/119905/why-does-my-regular-expression-work-in-x-but-not-in-y). ERE is used in `awk`.

##### Complexity of Matching with Backreferences

Standard regular expressions can be matched by a finite state automaton that runs in time linear in the size of the input. Regular expressions with backreferences need, in general, backtracking on the input, which can lead to exponential running time. This makes backreferences vulnerable to *regular expression denial-of-service* (ReDoS) attacks. [Go's regex](https://github.com/google/re2/wiki/Syntax) and [Rust's regex](https://docs.rs/regex/latest/regex/) do not support backreferences; this restriction allows matching in `O(m × n)`, where `m` is the size of the regular expression and `n` is the length of the input.

##### Programming Language Libraries

|   |Backreferences|Named capture groups|Named backreferences|Relative backreferences|
|---|:--|:--|:--|:--|
|[C regcomp (BRE)](https://pubs.opengroup.org/onlinepubs/9799919799/basedefs/V1_chap09.html)|`\1`|-|-|-|
|[C++ regex (BRE)](https://learn.microsoft.com/en-us/cpp/standard-library/regular-expressions-cpp)|`\1`|-|-|-|
|[C# (.NET regular expression)](https://docs.microsoft.com/en-us/dotnet/standard/base-types/grouping-constructs-in-regular-expressions#named_matched_subexpression)|`\1`, `\k{1}`|`(?<name>...)`, `(?'name'...)`| `\k<name>`|-|
|[Go regex/re2](https://github.com/google/re2/wiki/Syntax)|-|-|-|-|
|[Java regex](https://docs.oracle.com/en/java/javase/22/docs/api/java.base/java/util/regex/package-summary.html)| `\1`| `(?<name>...)` | `\k<name>`|-|
|[JavaScript](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Regular_expressions)|`\1` |`(?<name>...)`| `\k<name>` | - |
|[Perl](https://perldoc.perl.org/perlre.html)| `\1`, `\g1` |`(?<name>...)`, `(?'name'...)`| `\g{name}`, `\k<name>`, `\k'name'` | `\g{-1}` |
|[PHP PCRE regex](https://www.php.net/manual/en/regexp.reference.back-references.php)| `\1`, `\g{1}` |`(?P<name>...)`| `(?P=name)`, `\g{name}`, `\g'name'`, `\k<name>`, `\k'name'` | `\g{-1}`, `\g-1` |
|[Python re](https://docs.python.org/3/howto/regex.html)| `\1` |`(?P<name>...)`| `(?P=name)` | - |
|[Ruby regexp](https://ruby-doc.org/core-2.7.1/Regexp.html)|`\1` |`(?<name>...)`| `\g{name}`, `\g'name'`, `\k<name>`, `\k'name'` | `\k{-1}`, `\k-1` |
|[Rust regex](https://docs.rs/regex/latest/regex/)|-|-|-|-|
|[Swift NSRegularExpression](https://developer.apple.com/documentation/foundation/nsregularexpression)| `\1` | - |-|-|

##### Using Backreferences

When matching a string against a regular expression, `^` can be used to match the beginning of the string and `$` to match the end of the string. By enclosing a regular expression with `^` and `$`, only complete strings will match:

In [ ]:
%%javascript
const re = /^(ab|abc)*\1$/
element.append(re.test('abab'), ' ')
element.append(re.test('abcab'), ' ')
element.append(re.test('abcabc'), ' ')
element.append(re.test(''), ' ')
element.append(re.test('ababcababc'), ' ')
element.append(re.test('ababab'), ' ')

*Exercise.* Leave out `$` from the regular expression. Explain how the result changes!